# Phase 14 — a fold-0 screen for a larger network and a union-trained one

The working configuration (HRNet-W32 at 1024, five fold models, eight-view TTA,
the Phase 10 post-processing) scores a pooled five-fold PQ of 0.4127 and 0.36 on
the leaderboard. Reaching 0.40 there takes about 0.45 locally. The last three
phases moved PQ by about +0.002 each, so this screen only looks for large
effects: **at least +0.02 over run *e* on fold 0**, well beyond the 0.005 to 0.01
one retraining moves a fold.

Two candidates are trained at once, one per card, fold 0, twenty epochs each:

- **A. HRNet-W48.** HRNet won the comparison of four networks at 1024; W48
  widens every branch by half. If two frames of W48 do not fit a T4, it trains
  on one frame per batch with the gradients of two batches summed, so the update
  matches run *e*'s batch of two. A memory probe decides before training.
- **B. HRNet-W32 trained on the union of a frame's annotators.** Phase 12 traced
  the largest group of misses to long filaments covered only in part. The union
  asks for every pixel anyone drew. It is scored alone here; whether its map can
  extend the working configuration's regions is scored afterwards from the maps
  this notebook saves.

Both are scored as the submission is made: eight-view TTA, then the Phase 10
post-processing (2048, threshold 0.40, minimum area 400, rejoining 16, score cut
0.65), with Phase 12's breakdown of the misses. The references are run *e*
itself (0.4016) and its retraining in Phase 13 (0.4030), so no control is
trained here.

**Before running**, in the notebook settings:

1. Accelerator: **GPU T4 x2** (one run per card)
2. Internet: **on**
3. Inputs: the competition data
4. **Save Version** with *Save output* on

Expected wall clock: two and a half to three hours (W48 trains more slowly than
W32).

**Output.** Each run's checkpoint and logs under `w48/` and `union/`, its
eight-view fold 0 maps under `maps/`, and the scores and miss breakdowns under
`scores/`.

**Re-running after a failure.** Attach this notebook's earlier version as an
input. Finished training and maps are copied back and skipped.

## 1. Clone the repository and put it on the path

Cloned into `/tmp`, not `/kaggle/working`, so the clone does not count against
the 500 output files.

In [ ]:
import sys

REPO_URL = "https://github.com/KeiichiIto1978/Solar_Filament_Segmentation_Challenge_2026.git"
REF = "feat/phase14-screen"  # branch or commit hash
CHECKOUT = "/tmp/repo"

!rm -rf {CHECKOUT}
!git clone -q --branch {REF} {REPO_URL} {CHECKOUT}
!cd {CHECKOUT} && git log --oneline -1

# Only what Kaggle does not already have; torch stays as it is.
!pip install -q "segmentation-models-pytorch>=0.5"

if f"{CHECKOUT}/src" not in sys.path:
    sys.path.insert(0, f"{CHECKOUT}/src")

In [ ]:
import segmentation_models_pytorch as smp
import torch

import filament

print("filament", filament.__version__)
print("smp", smp.__version__)
print("torch", torch.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.device_count())

## 2. Point the package at the competition data

In [ ]:
import os
from pathlib import Path

from filament.paths import load_paths

ANNOTATION_NAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

candidates = sorted(Path("/kaggle/input").glob(f"**/{ANNOTATION_NAME}"))
if not candidates:
    raise SystemExit(
        "Could not find the annotation file under /kaggle/input. "
        "Attach the competition data to this notebook first."
    )

os.environ["MAGFILO_ROOT"] = str(candidates[0].parent.parent)
paths = load_paths().require_dataset()
print("MAGFILO_ROOT =", os.environ["MAGFILO_ROOT"])
print("train images:", len(list(paths.train_images.glob("*.jpeg"))))

## 3. The two runs

In [ ]:
import json
import shutil
import subprocess
import sys
from dataclasses import replace

from filament.training.config import TrainConfig

WORKING = Path("/kaggle/working")
# Two frames of W48 at 1024 may not fit a T4. A few training steps in a fresh
# process decide, before any training starts; the notebook itself never touches
# the card, so the runs get all of its memory.
PROBE = """
import torch
from filament.models.segmentation import DiceBceLoss, ModelConfig, build_model
model = build_model(ModelConfig(encoder_name="tu-hrnet_w48", encoder_weights=None)).cuda()
optimiser = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler("cuda")
loss = DiceBceLoss()
for _ in range(3):
    image = torch.rand(2, 2, 1024, 1024, device="cuda")
    target = (torch.rand(2, 1, 1024, 1024, device="cuda") > 0.99).float()
    with torch.amp.autocast("cuda"):
        value = loss(model(image), target)
    optimiser.zero_grad()
    scaler.scale(value).backward()
    scaler.step(optimiser)
    scaler.update()
print(f"peak {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")
"""
W48_FITS = True
if torch.cuda.is_available():
    probe = subprocess.run(
        [sys.executable, "-c", PROBE],
        env=os.environ | {"CUDA_VISIBLE_DEVICES": "0", "PYTHONPATH": f"{CHECKOUT}/src"},
        capture_output=True,
        text=True,
    )
    W48_FITS = probe.returncode == 0
    print("W48, two frames:", probe.stdout.strip() or probe.stderr.strip().splitlines()[-1])
W48_CONFIG = "hrnet_w48" if W48_FITS else "hrnet_w48_accum"
print(f"W48 trains with configs/phase14/{W48_CONFIG}.yaml")

RUNS = {
    "w48": f"{CHECKOUT}/configs/phase14/{W48_CONFIG}.yaml",
    "union": f"{CHECKOUT}/configs/phase14/hrnet_w32_union.yaml",
}
configs = {
    name: replace(TrainConfig.from_yaml(path), num_workers=2, output_dir=WORKING / name)
    for name, path in RUNS.items()
}
for name, config in configs.items():
    print(
        f"{name:8s} {config.encoder} {config.image_size}px batch {config.batch_size} "
        f"(accumulating {config.accumulation_steps}) epochs {config.epochs} "
        f"union {config.union_targets} fold {config.fold}"
    )

# Kaggle empties /kaggle/working when a session starts, so an earlier version's
# work is only reachable through the inputs. A run is restored only if its stored
# configuration is the one this notebook would train, and its maps and scores
# only from that same earlier version: a folder name alone could bring back the
# results of another configuration as if they were this one's.
RUN_FILES = ("best.pt", "config.yaml", "history.json", "train.log")
IGNORED = {"output_dir", "num_workers"}


def _same_run(stored: Path, config: TrainConfig) -> bool:
    restored = TrainConfig.from_yaml(stored).to_dict()
    current = config.to_dict()
    return all(restored[key] == current[key] for key in current if key not in IGNORED)


for name, config in configs.items():
    config.output_dir.mkdir(parents=True, exist_ok=True)
    for source in Path("/kaggle/input").glob(f"**/{name}/best.pt"):
        if not (source.parent / "config.yaml").exists() or not _same_run(
            source.parent / "config.yaml", config
        ):
            print(f"{name}: {source.parent} holds another configuration; not restored")
            continue
        for item in RUN_FILES:
            if (source.parent / item).exists() and not (config.output_dir / item).exists():
                shutil.copy(source.parent / item, config.output_dir / item)
        print(f"{name}: restored from {source.parent}")
        version = source.parent.parent
        for folder in ("maps", "scores"):
            (WORKING / folder).mkdir(exist_ok=True)
            for item in (version / folder).glob(f"{name}_*"):
                if item.is_file() and not (WORKING / folder / item.name).exists():
                    shutil.copy(item, WORKING / folder / item.name)
                    print(f"restored {folder}/{item.name}")
        break
for folder in ("maps", "scores"):
    (WORKING / folder).mkdir(exist_ok=True)

## 4. Training, one run per card

Each run is `scripts/train.py` in its own process, pinned to one card with
`CUDA_VISIBLE_DEVICES`, so that each seeds its own random state.

In [ ]:
import subprocess
import sys
import threading
import time

gpu_count = torch.cuda.device_count()
pending = [name for name in RUNS if not (configs[name].output_dir / "best.pt").exists()]
QUEUES = (
    {gpu: pending[gpu :: max(gpu_count, 1)] for gpu in range(max(gpu_count, 1))}
    if gpu_count
    else {}
)
print(f"{gpu_count} GPU(s):", QUEUES)
print_lock = threading.Lock()
failures = {}


def run_queue(gpu, names):
    # An exception in a thread is otherwise printed and forgotten, and the
    # notebook would go on to fail later on a missing checkpoint.
    try:
        _run_queue(gpu, names)
    except Exception as error:
        with print_lock:
            print(f"[gpu{gpu}] stopped: {error!r}")
        failures.update({name: repr(error) for name in names if name not in failures})
        raise


def _run_queue(gpu, names):
    for name in names:
        config = configs[name]
        command = [
            sys.executable,
            f"{CHECKOUT}/scripts/train.py",
            "--config",
            RUNS[name],
            "--output-dir",
            str(config.output_dir),
            "--num-workers",
            str(config.num_workers),
        ]
        environment = os.environ | {
            "CUDA_VISIBLE_DEVICES": str(gpu),
            "PYTHONPATH": f"{CHECKOUT}/src",
        }
        with print_lock:
            print(f"[gpu{gpu}] {name}: started")
        started = time.perf_counter()
        with open(config.output_dir / "train.log", "w") as log:
            process = subprocess.Popen(
                command,
                cwd=CHECKOUT,
                env=environment,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
            )
            for line in process.stdout:
                log.write(line)
                log.flush()
                if "Epoch" in line or "Error" in line or "Traceback" in line:
                    with print_lock:
                        print(f"[gpu{gpu}] {name}: {line.rstrip()}")
            exit_code = process.wait()
        with print_lock:
            minutes = (time.perf_counter() - started) / 60
            print(f"[gpu{gpu}] {name}: exit {exit_code} after {minutes:.1f} min")
        if exit_code != 0:
            failures[name] = exit_code


threads = [threading.Thread(target=run_queue, args=item, daemon=True) for item in QUEUES.items()]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
for name, exit_code in failures.items():
    tail = (configs[name].output_dir / "train.log").read_text().splitlines()[-20:]
    print(f"{name} FAILED with exit code {exit_code}:", *tail, sep="\n")
if failures:
    raise SystemExit(f"Training failed for {sorted(failures)}.")

## 5. Eight-view maps of fold 0

The same TTA as the submission: each frame in the eight flips and quarter
turns, turned back and averaged. Only the first output channel, the mask, is
read.

In [ ]:
from filament.data.image import load_grayscale
from filament.data.split import load_fold
from filament.evaluation import predict_probability_tta
from filament.postprocess.search import save_map_bundle
from filament.training.loop import HISTORY_NAME, load_checkpoint

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
val_stems = sorted(load_fold(0, paths.splits_dir).val)
for name, config in configs.items():
    path = WORKING / "maps" / f"{name}_fold0.npz"
    if path.exists():
        print(f"{name}: maps already written")
        continue
    model, stored = load_checkpoint(config.output_dir / "best.pt", device=DEVICE)
    started = time.perf_counter()
    maps = {
        stem: predict_probability_tta(
            model,
            load_grayscale(paths.train_images / f"{stem}.jpeg"),
            size=config.image_size,
            device=DEVICE,
        )
        for stem in val_stems
    }
    save_map_bundle(maps, path)
    history = json.loads((config.output_dir / HISTORY_NAME).read_text())
    best = min(history, key=lambda item: item["val_loss"])
    print(
        f"{name}: {len(maps)} maps in {time.perf_counter() - started:.0f}s; "
        f"best epoch {best['epoch']} of {len(history)}, val loss {best['val_loss']:.4f}"
    )
    del model, maps
    torch.cuda.empty_cache()

## 6. Scores and the misses

`scripts/sweep_maps.py` scores each run's maps under the Phase 10 setting and,
for reference, the same without the score cut; `scripts/miss_breakdown.py`
sorts every annotated filament of fold 0 by outcome, as in Phase 12. Both run
as their own processes on the CPU.

In [ ]:
import pandas as pd

from filament.metrics.misses import CATEGORIES
from filament.postprocess.search import SweepResult

PHASE10 = {
    "resolution": 2048,
    "threshold": 0.4,
    "min_area": 400,
    "join_gap": 16.0,
    "join_offset": 12.0,
    "min_score": 0.65,
}
SETTINGS = [PHASE10, PHASE10 | {"min_score": 0.0}]
# Run e on fold 0 under the same TTA and post-processing (Phase 10, from Phase 9's maps).
RUN_E_FOLD0 = {"pq": 0.4016}
scores_dir = WORKING / "scores"
(scores_dir / "settings.json").write_text(json.dumps(SETTINGS))
(scores_dir / "phase10.json").write_text(json.dumps(PHASE10))
environment = os.environ | {"PYTHONPATH": f"{CHECKOUT}/src"}

jobs = []
for name in configs:
    maps = str(WORKING / "maps" / f"{name}_fold0.npz")
    sweep_out, miss_out = scores_dir / f"{name}_sweep.json", scores_dir / f"{name}_misses.csv"
    if not sweep_out.exists():
        jobs.append(
            subprocess.Popen(
                [
                    sys.executable,
                    f"{CHECKOUT}/scripts/sweep_maps.py",
                    "--maps",
                    maps,
                    "--fold",
                    "0",
                    "--settings",
                    str(scores_dir / "settings.json"),
                    "--output",
                    str(sweep_out),
                ],
                cwd=CHECKOUT,
                env=environment,
            )
        )
    if not miss_out.exists():
        jobs.append(
            subprocess.Popen(
                [
                    sys.executable,
                    f"{CHECKOUT}/scripts/miss_breakdown.py",
                    "--maps",
                    maps,
                    "--fold",
                    "0",
                    "--setting",
                    str(scores_dir / "phase10.json"),
                    "--output",
                    str(miss_out),
                ],
                cwd=CHECKOUT,
                env=environment,
            )
        )
if any(job.wait() for job in jobs):
    raise SystemExit("A scoring process failed; its output is above.")

rows = {}
for name in configs:
    sweep = SweepResult.from_records(
        json.loads((scores_dir / f"{name}_sweep.json").read_text())["points"]
    )
    misses = pd.read_csv(scores_dir / f"{name}_misses.csv")
    for point in sweep.points:
        label = f"{name}, score cut {point.setting.values['min_score']}"
        rows[label] = {
            "pq": point.pq.pq,
            "sq": point.pq.sq,
            "rq": point.pq.rq,
            "tp": point.pq.tp,
            "fp": point.pq.fp,
            "fn": point.pq.fn,
        }
    counts = misses["outcome"].value_counts()
    rows[f"{name}, score cut 0.65"] |= {
        category: int(counts.get(category, 0)) for category in CATEGORIES
    }
table = pd.DataFrame(rows).T
# Fold 0, eight-view TTA and the Phase 10 post-processing: run e from Phase 9's
# maps, and its retraining as the control of Phase 13 with that run's misses.
REFERENCES = {"run e": 0.4016, "run e retrained": 0.4030}
REFERENCE_MISSES = {
    "unseen": 137,
    "below_threshold": 129,
    "dropped_area": 57,
    "dropped_score": 34,
    "too_large": 127,
    "fragmented": 52,
    "too_small": 179,
    "shape": 72,
}
GATE = 0.02
reference = max(REFERENCES.values())
for name in configs:
    row = rows[f"{name}, score cut 0.65"]
    gain = row["pq"] - reference
    print(
        f"{name}: PQ {row['pq']:.4f} (SQ {row['sq']:.4f}, RQ {row['rq']:.4f}); "
        f"{gain:+.4f} against the better reference -> "
        f"{'PASSES' if gain >= GATE else 'below'} the +{GATE} gate"
    )
    changes = ", ".join(
        f"{category} {row[category] - count:+d}" for category, count in REFERENCE_MISSES.items()
    )
    print(f"  misses against the retrained run e: {changes}")
(scores_dir / "summary.json").write_text(
    json.dumps({"rows": rows, "references": REFERENCES, "commit": REF}, indent=1)
)
table.round(4)

In [ ]:
# Kaggle keeps at most 500 output files and drops the rest without an error.
saved = [item for item in WORKING.rglob("*") if item.is_file()]
total_mb = sum(item.stat().st_size for item in saved) / 1e6
print(f"{len(saved)} files in /kaggle/working, {total_mb:.0f} MB")
if len(saved) > 500:
    raise SystemExit(f"{len(saved)} output files; Kaggle would silently drop some of them.")

## 7. What to record

- both runs' PQ, SQ, RQ, TP, FP, FN under the Phase 10 setting, with and
  without the score cut, their best epoch and validation loss, and which W48
  configuration the memory probe chose
- their miss breakdowns against the retrained run *e*
- whether either passes the +0.02 gate
- the commit this notebook cloned